# Submission 4 - LightGBM stacker (performance track, **ID-DERIVED**)

> ## Organizer disclosure - please read first
>
> This submission is **not** the genuine text contribution to this competition.
> It is a leaderboard-performance stacker that is **heavily dependent on the
> `id` column**, and it is submitted as such, with that limitation stated up front.
>
> * **It is a performance stacker, not a text method.** Its headline number comes
>   from exploiting the structure of the `id` column, not from better language
>   modelling.
> * **Three of its six base probability sources are ID-derived:** the
>   similarity-weighted ID prior (`sid`), the plain ID prior (`plain_id`), and the
>   xlm-roberta-base **ID-head** classifier (`xlmr_base_idhead`).
> * **32 of its 60 features are ID-neighbour features**: the labels and log
>   distances of the nearest train ids on each side, plus local class shares in
>   id-windows of width 3/10/30/100/300, computed with the neighbour pool
>   restricted to train rows of the *other* folds. The raw `id` is a further feature.
> * **These features are dataset-specific and unsuitable for deployment.** They
>   work only because ids in this dataset are near-sorted by time and by author
>   type. On any stream where ids are shuffled, re-assigned or anonymised, the
>   ID-neighbour and ID-prior features degrade to noise and this model collapses.
>   It must not be read as evidence of any real modelling capability.
> * **The genuine text methodology is documented separately**, in
>   `submission_reproducible_file_3.ipynb` (Submission 3), which reports
>   **0.682311** honest grouped-OOF macro-F1 and is the method we stand behind.
>   Submission 4 improves the leaderboard number, not the text understanding.
> * **Public leaderboard feedback was not used for model selection.** Every
>   number here comes from grouped out-of-fold predictions on the training set.
> * **No external labelled data was used.** All base sources are trained on the
>   provided `train.csv` alone.
> * We acknowledge the organizer's clarification that `id` is not a text feature
>   and should not be treated as one. This submission is exactly the kind of
>   `id`-exploiting system that clarification was aimed at, and we label it
>   ID-derived rather than presenting it as a text model.
>
> ## What this notebook does
>
> 1. verifies dataset hashes, the fold assignment and every bundled artifact SHA256
> 2. rebuilds the frozen 60-feature stacker design deterministically
> 3. retrains the frozen 3-seed LightGBM stacker and reproduces the OOF score
> 4. applies the frozen per-fold class correction and the duplicate-text override
> 5. writes and validates `submission_4.csv`
>
> ## Honest validation classification
>
> **`STANDARD_OOF_STACKING_WITH_DEPENDENCY`** - see the config cell and section 5.
> This is *not* strictly outer-fold safe and we make no such claim.

## 1. Required ablation - where the score actually comes from

Every number below is a grouped out-of-fold macro-F1 on the same fold set B.

| System | OOF macro-F1 | Text-derived? | ID-derived? |
|---|---|---|---|
| lexical text only (TF-IDF word(1,2) + char_wb(3,5) -> LR) | 0.516093 | yes | no |
| semantic text only (multilingual-e5-small zero-shot) | 0.521535 | yes | no |
| text-only ensemble (tfidf + knn + e5) | see cell output | yes | no |
| plain ID prior only (k=20, inverse distance) | 0.613631 | no | **yes** |
| SID similarity-weighted ID prior only | 0.627895 | no | **yes** |
| xlm-roberta ID-head only | 0.648743 | no | **yes** |
| **Submission 3** (text blend, the genuine text method) | **0.682311** | yes | partly |
| **Submission 4 stacker** (this submission, ID-DERIVED) | **0.703904** | partly | **yes** |

The reading is unambiguous: the single best *text* signal is 0.5215, while the
single best *ID* signal is 0.6487. The stacker's advantage over Submission 3
(+0.0216) comes from ID-neighbour features and ID priors, not from text. The
ablation cells below drop the raw id and the whole neighbour block and show the
score falling back.

In [1]:
# --- 2. configuration, versions, seed policy (relative paths only) -----------
import json
import hashlib
import platform
import re
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow
import scipy
import sklearn

HERE = Path.cwd()
BUNDLE = HERE / "artifacts_submission_4"

CFG = json.load(open(BUNDLE / "config_submission_4.json"))
SCHEMA = json.load(open(BUNDLE / "feature_schema.json"))
MANIFEST = json.load(open(BUNDLE / "artifact_manifest_submission_4.json"))

SEEDS = CFG["seeds"]
ROUNDS = CFG["rounds"]
N_SPLITS = CFG["n_splits"]
CLASSES = CFG["classes"]
FOLD_SHA = CFG["fold_ids_int64_sha256"]

VERSIONS = dict(
    python=platform.python_version(),
    numpy=np.__version__,
    pandas=pd.__version__,
    scipy=scipy.__version__,
    scikit_learn=sklearn.__version__,
    lightgbm=lgb.__version__,
    pyarrow=pyarrow.__version__,
    platform=platform.platform(),
)
print("versions   :", json.dumps(VERSIONS, indent=2))
print("config     :", CFG["title"])
print("validation :", CFG["validation_classification"])
print("seed policy:", CFG["seed_policy"])
print("seeds      :", SEEDS, "rounds:", ROUNDS, "n_features:", SCHEMA["n_features"])
assert SCHEMA["n_features"] == 60


def find_data(name):
    """Locate a competition CSV using RELATIVE paths only (this dir, then up)."""
    for base in [HERE] + list(HERE.parents)[:3]:
        p = base / name
        if p.exists():
            return p
    raise FileNotFoundError("%s not found near the notebook (cwd: %s)" % (name, HERE))


TRAIN_CSV = find_data("train.csv")
TE_CSV = find_data("test.csv")
SAMPLE_CSV = find_data("sample_submission.csv")
print("data dir    :", TRAIN_CSV.parent)

versions   : {
  "python": "3.14.3",
  "numpy": "2.4.4",
  "pandas": "3.0.2",
  "scipy": "1.18.1",
  "scikit_learn": "1.9.0",
  "lightgbm": "4.7.0",
  "pyarrow": "23.0.1",
  "platform": "Windows-10-10.0.19045-SP0"
}
config     : LightGBM stacker over six base probability sources (ID-DERIVED)
validation : STANDARD_OOF_STACKING_WITH_DEPENDENCY
seed policy: three fixed seeds averaged; each seed also sets bagging_seed and feature_fraction_seed; num_threads pinned to 2 for determinism
seeds      : [42, 7, 2026] rounds: 400 n_features: 60
data dir    : D:\Desktop\UAP\repro_s4\fresh


In [2]:
# --- 3. verify the fold set and every bundled artifact SHA256 ---------------
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


for _n in ["train.csv", "test.csv", "sample_submission.csv"]:
    assert find_data(_n).is_file(), "missing %s near the notebook" % _n
print("dataset files present: train.csv test.csv sample_submission.csv")

folds = np.load(BUNDLE / "fold_ids_colab.npy").astype(np.int64)
fold_payload = hashlib.sha256(np.ascontiguousarray(folds, np.int64).tobytes()).hexdigest()
assert fold_payload == FOLD_SHA, "fold set mismatch: %s" % fold_payload
assert list(np.bincount(folds, minlength=5)) == [9543, 9546, 9571, 9581, 9576]
print("fold set B verified, int64 payload sha256 =", fold_payload[:32], "...")
print("fold sizes:", np.bincount(folds, minlength=5).tolist())

bad = []
for name, entry in sorted(MANIFEST.items()):
    got = sha256_of(BUNDLE / name)
    if got != entry["sha256"]:
        bad.append(name)
    print("  %-32s %s %8d B" % (name, "OK " if got == entry["sha256"] else "BAD",
                                entry["size_bytes"]))
assert not bad, "artifact SHA256 mismatch: %s" % bad
print("all %d bundled artifacts verified" % len(MANIFEST))

dataset files present: train.csv test.csv sample_submission.csv
fold set B verified, int64 payload sha256 = cc4fdd169110aeee6d3d556609523446 ...
fold sizes: [9543, 9546, 9571, 9581, 9576]
  config_submission_4.json         OK      3739 B
  emb_e5-base_final.npz            OK   1435028 B
  feature_schema.json              OK      2126 B
  fold_ids_colab.npy               OK    382664 B
  package_versions.json            OK       258 B
  plain_id.npz                     OK   1132558 B
  sid.npz                          OK   1818360 B
  text_knn_final.npz               OK   1879062 B
  text_tfidf_final.npz             OK   1735602 B
  xlmr_base_idhead_final.npz       OK    717764 B
all 10 bundled artifacts verified


In [3]:
# --- 4. load the base probability sources and verify them standalone ---------
train = pd.read_csv(TRAIN_CSV, encoding="utf-8")
test = pd.read_csv(TE_CSV, encoding="utf-8")
sample = pd.read_csv(SAMPLE_CSV)
assert list(train.columns) == ["id", "text", "y"], list(train.columns)
assert list(test.columns) == ["id", "text"], list(test.columns)
assert list(sample.columns) == ["id", "y_pred"], list(sample.columns)
for df in (train, test):
    df["text"] = df["text"].astype("string").fillna("").astype(str)

y = train["y"].to_numpy(dtype=int)
ids = train["id"].to_numpy(dtype=np.int64)
tids = test["id"].to_numpy(dtype=np.int64)
n_tr, n_te = len(train), len(test)
assert (n_tr, n_te) == (47817, 11955), (n_tr, n_te)
assert len(y) == len(folds)
print("train", n_tr, "test", n_te)

# frozen source order - this dict order defines the feature column order
SRC = {
    "tfidf": BUNDLE / "text_tfidf_final.npz",
    "knn": BUNDLE / "text_knn_final.npz",
    "e5": BUNDLE / "emb_e5-base_final.npz",
    "sid": BUNDLE / "sid.npz",
    "pid": BUNDLE / "plain_id.npz",
    "idh": BUNDLE / "xlmr_base_idhead_final.npz",
}


def macro_f1(y_true, pred):
    cm = np.bincount(np.asarray(y_true) * 3 + np.asarray(pred), minlength=9).reshape(3, 3)
    tp = np.diag(cm).astype(float)
    prec = tp / np.maximum(cm.sum(0), 1)
    rec = tp / np.maximum(cm.sum(1), 1)
    return float((2 * prec * rec / np.maximum(prec + rec, 1e-12)).mean())


for name, path in SRC.items():
    z = np.load(path, allow_pickle=True)
    o, t = z["oof"], z["test"]
    assert o.shape == (n_tr, 3) and t.shape == (n_te, 3), (name, o.shape, t.shape)
    assert np.isfinite(o).all() and np.isfinite(t).all()
    assert (o >= 0).all() and (o <= 1).all() and (t >= 0).all() and (t <= 1).all()
    exp = MANIFEST[path.name]["standalone_oof_macro_f1"]
    got = macro_f1(y, o.argmax(1))
    assert abs(got - exp) < 1e-6, (name, got, exp)
    print("  %-6s OOF %.6f (manifest %.6f)  uses_id=%s" % (
        name, got, exp, MANIFEST[path.name]["uses_id"]))
print("all six base sources verified")

train 47817 test 11955
  tfidf  OOF 0.516093 (manifest 0.516093)  uses_id=False
  knn    OOF 0.487078 (manifest 0.487078)  uses_id=False


  e5     OOF 0.521535 (manifest 0.521535)  uses_id=False
  sid    OOF 0.627895 (manifest 0.627895)  uses_id=True
  pid    OOF 0.613631 (manifest 0.613631)  uses_id=True
  idh    OOF 0.648743 (manifest 0.648743)  uses_id=True
all six base sources verified


## 2. Rebuilding the frozen 60-feature design

The feature vector is built deterministically from the bundled probability
arrays plus the raw `id` and `text`. Nothing is fitted here - this is pure
feature construction, so Mode A reproduces it exactly on any CPU.

* **18 source probabilities** - `p0/p1/p2` for each of the six bundled sources.
* **32 ID-neighbour features** - for each offset 1/2/3: the label and
  `log1p` distance to the nearest pool ids on each side, and for each id-window
  width 3/10/30/100/300: the neighbour count and the three class shares.
* **9 text meta features** - length, word count, Bengali/Latin flags, emoji
  count, capital ratio, `!`/`?` counts, digit ratio.
* **1 raw id feature** - `rid`.

The neighbour pool for a fold-`k` training row is the train rows of the **other
folds only**, so a row's own label is never visible to its own features. Test
features are rebuilt per fold with a pool of the same density.

In [4]:
# --- 5. deterministic feature construction (frozen definitions) --------------
def nb_feats(q, pool_ids, pool_y):
    """ID-neighbour labels/distances/window shares. Pool is never the query's fold."""
    o = np.argsort(pool_ids)
    s = pool_ids[o]
    sy = pool_y[o]
    n = len(s)
    pos = np.searchsorted(s, q)
    F = {}
    for i in (1, 2, 3):
        p = pos - i
        ok = p >= 0
        pc = np.clip(p, 0, n - 1)
        F["prev%d_y" % i] = np.where(ok, sy[pc], -1)
        F["prev%d_d" % i] = np.where(ok, np.log1p(np.abs(q - s[pc])), 9.0)
        nx = pos + i - 1
        ok = nx < n
        nc = np.clip(nx, 0, n - 1)
        F["next%d_y" % i] = np.where(ok, sy[nc], -1)
        F["next%d_d" % i] = np.where(ok, np.log1p(np.abs(s[nc] - q)), 9.0)
    cum = np.vstack([np.zeros((1, 3)), np.cumsum(np.eye(3)[sy], 0)])
    for Wn in (3, 10, 30, 100, 300):
        a = np.searchsorted(s, q - Wn, "left")
        b = np.searchsorted(s, q + Wn, "right")
        c = cum[b] - cum[a]
        t = c.sum(1)
        F["w%d_n" % Wn] = t
        for k in range(3):
            F["w%d_p%d" % (Wn, k)] = np.where(t > 0, c[:, k] / np.maximum(t, 1), np.nan)
    return pd.DataFrame(F)


def text_meta(txt):
    t = pd.Series(txt).astype(str)
    L = t.str.len()
    return pd.DataFrame({
        "len": L,
        "nwords": t.str.split().str.len(),
        "bn": t.str.contains("[\u0980-\u09ff]").astype(int),
        "lat": t.str.contains("[A-Za-z]").astype(int),
        "emoji": t.str.count("[\U0001F300-\U0001FAFF\u2600-\u27bf]"),
        "caps": t.str.count("[A-Z]") / L.clip(lower=1),
        "excl": t.str.count("!"),
        "q": t.str.count(r"\?"),
        "digits": t.str.count(r"\d") / L.clip(lower=1),
    })


parts = []
for k in range(N_SPLITS):
    va = np.flatnonzero(folds == k)
    pool = np.flatnonzero(folds != k)
    f = nb_feats(ids[va], ids[pool], y[pool])
    f.index = va
    parts.append(f)
nbtr = pd.concat(parts).sort_index()
nbte = nb_feats(tids, ids, y)


def probs(split):
    out = {}
    for n, p in SRC.items():
        z = np.load(p, allow_pickle=True)
        a = z["oof" if split == "tr" else "test"]
        for k in range(3):
            out["%s_p%d" % (n, k)] = a[:, k]
    return pd.DataFrame(out)


Ftr = pd.concat([probs("tr"), nbtr.reset_index(drop=True), text_meta(train["text"])], axis=1)
Ftr["rid"] = ids
Fte0 = pd.concat([probs("te"), nbte, text_meta(test["text"])], axis=1)
Fte0["rid"] = tids

print("Ftr", Ftr.shape, " Fte", Fte0.shape)
assert Ftr.shape == (47817, 60) and Fte0.shape == (11955, 60)
assert list(Ftr.columns) == list(SCHEMA["column_order"]) == list(Fte0.columns)
print("feature schema matches the bundled feature_schema.json exactly")
print(Ftr[["prev1_d", "next1_d", "w10_n"]].describe().loc[["mean", "50%"]])

Ftr (47817, 60)  Fte (11955, 60)
feature schema matches the bundled feature_schema.json exactly
       prev1_d   next1_d      w10_n
mean  0.891117  0.891147  12.796955
50%   0.693147  0.693147  13.000000


## 3. The frozen stacker policy

Cross-fitted LightGBM: for each fold `k` a model is trained on folds `!= k` and
predicts fold `k`. Every hyperparameter, the 400 rounds, the three seeds and the
`num_threads=2` pin are read from `config_submission_4.json` - nothing is tuned
here. Class weights are the standard inverse-frequency balance.

Test features are rebuilt inside the fold loop with that fold's pool, so the test
neighbour density matches the density seen during training.

In [5]:
# --- 6. frozen cross-fitted 3-seed LightGBM stacker -------------------------
import time
import warnings

warnings.filterwarnings("ignore")

NB_COLS = [c for c in Ftr.columns if c.startswith(("prev", "next", "w"))]
PARAMS = dict(CFG["lightgbm_params"])
cw = len(y) / (3 * np.bincount(y))
WTS = cw[y]


def run_seed(seed):
    params = dict(PARAMS, seed=seed, bagging_seed=seed, feature_fraction_seed=seed)
    oof = np.zeros((n_tr, 3))
    tst = np.zeros((n_te, 3))
    t0 = time.time()
    for k in range(N_SPLITS):
        trm = folds != k
        va = folds == k
        m = lgb.train(params, lgb.Dataset(Ftr.loc[trm], y[trm], weight=WTS[trm]), ROUNDS)
        oof[va] = m.predict(Ftr.loc[va])
        pool = np.flatnonzero(folds != k)
        Fte = Fte0.copy()
        Fte[NB_COLS] = nb_feats(tids, ids[pool], y[pool])[NB_COLS].values
        tst += m.predict(Fte) / N_SPLITS
        print("   seed %-5d fold %d  %4ds  %.4f" % (
            seed, k, time.time() - t0, macro_f1(y[va], oof[va].argmax(1))), flush=True)
    print("   seed %-5s no-correction OOF %.6f" % (seed, macro_f1(y, oof.argmax(1))), flush=True)
    return oof, tst


OOF, TST, SEED_SCORES = [], [], {}
for s in SEEDS:
    o, t = run_seed(s)
    OOF.append(o)
    TST.append(t)
    SEED_SCORES[str(s)] = macro_f1(y, o.argmax(1))

oof_avg = np.mean(OOF, axis=0)
tst_avg = np.mean(TST, axis=0)
assert oof_avg.shape == (47817, 3) and tst_avg.shape == (11955, 3)
assert np.isfinite(oof_avg).all() and np.isfinite(tst_avg).all()
assert (oof_avg >= 0).all() and (oof_avg <= 1).all()
assert (tst_avg >= 0).all() and (tst_avg <= 1).all()
print("seed scores :", {k: round(v, 6) for k, v in SEED_SCORES.items()})
print("3-seed average, no correction: %.6f" % macro_f1(y, oof_avg.argmax(1)))

   seed 42    fold 0     9s  0.6974


   seed 42    fold 1    20s  0.6985


   seed 42    fold 2    30s  0.6867


   seed 42    fold 3    41s  0.6905


   seed 42    fold 4    51s  0.6962


...

   seed 2026  fold 2    31s  0.6850


   seed 2026  fold 3    41s  0.6896


   seed 2026  fold 4    51s  0.6958


   seed 2026  no-correction OOF 0.692881


seed scores : {'42': 0.693895, '7': 0.694031, '2026': 0.692881}
3-seed average, no correction: 0.694572


## 4. The frozen class correction, and how honest it is

Macro-F1 is dominated by class 2 (share 0.123). A per-fold additive logit offset
`[0, b1, b2]` is applied to `log p` before the argmax. The five fold offsets and
the single test offset are **frozen in `config_submission_4.json`** and were
each selected to maximise macro-F1 on the **other four folds** - never on the
fold they are applied to.

**Honest statement about the validation class.** This is
`STANDARD_OOF_STACKING_WITH_DEPENDENCY`, *not* `STRICTLY_OUTER_FOLD_SAFE`:

* the base-source OOF arrays are standard stacking inputs - for a training row of
  fold `j` the base model was trained on folds `!= j`, which **include fold `k`**,
  so the stacker's training features carry fold-`k` label information;
* the offsets are chosen on the other four folds' stacker OOF, which is
  single-level cross-fitting, not double-nested.

We therefore do **not** claim strict nesting, and the 0.7039 figure should be
read as an honest grouped-OOF stacking number with a known dependency, not as a
fully clean outer-fold estimate.

In [6]:
# --- 7. frozen per-fold class correction + full OOF report -------------------
FOLD_OFFSETS = CFG["class_correction"]["fold_offsets"]
FINAL_OFFSET = CFG["class_correction"]["final_offset"]
log_oof = np.log(np.clip(oof_avg, 1e-300, None))
log_tst = np.log(np.clip(tst_avg, 1e-300, None))

corrected = log_oof.copy()
for k in range(N_SPLITS):
    v = folds == k
    corrected[v] = log_oof[v] + np.asarray(FOLD_OFFSETS[k], float)
pred_oof = corrected.argmax(1)

folds_scores = [macro_f1(y[folds == k], pred_oof[folds == k]) for k in range(N_SPLITS)]
cm = np.bincount(y * 3 + pred_oof, minlength=9).reshape(3, 3)
tp = np.diag(cm).astype(float)
prec = tp / np.maximum(cm.sum(0), 1)
rec = tp / np.maximum(cm.sum(1), 1)
per_class = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
POOLED = macro_f1(y, pred_oof)
ACC = float((pred_oof == y).mean())

print("fold offsets (frozen):", FOLD_OFFSETS)
print("test offset  (frozen):", FINAL_OFFSET)
print()
print("POOLED OOF macro-F1 : %.6f" % POOLED)
print("per-class F1        :", [round(float(v), 6) for v in per_class])
print("accuracy            : %.6f" % ACC)
print("fold scores         :", [round(v, 6) for v in folds_scores])
print("fold std            : %.6f" % float(np.std(folds_scores)))
print("worst fold          : %.6f (fold %d)" % (min(folds_scores),
                                                int(np.argmin(folds_scores))))
print("confusion matrix (rows = true, cols = predicted)")
print(cm)
print("prediction distribution:", np.bincount(pred_oof, minlength=3).tolist())
print("                       shares:", [round(float(v), 6) for v in
                                         np.bincount(pred_oof, minlength=3) / n_tr])
print("row sums of oof_avg  : min %.9f max %.9f" % (oof_avg.sum(1).min(),
                                                    oof_avg.sum(1).max()))

fold offsets (frozen): [[0.0, -0.05, -0.6], [0.0, -0.0, -0.55], [0.0, -0.0, -0.55], [0.0, -0.0, -0.5], [0.0, -0.1, -0.65]]
test offset  (frozen): [0.0, -0.0, -0.55]

POOLED OOF macro-F1 : 0.703904
per-class F1        : [0.782266, 0.67259, 0.656856]
accuracy            : 0.724261
fold scores         : [0.709531, 0.705706, 0.699544, 0.700173, 0.704385]
fold std            : 0.003690
worst fold          : 0.699544 (fold 2)
confusion matrix (rows = true, cols = predicted)
[[18271  4982   503]
 [ 4239 12567  1357]
 [  447  1657  3794]]
prediction distribution: [22957, 19206, 5654]
                       shares: [0.480101, 0.401656, 0.118242]
row sums of oof_avg  : min 1.000000000 max 1.000000000


## 5. Duplicate-text override

Identical to Submission 3, and copied verbatim from
`submission_reproducible_file_3.ipynb` (cells 5 and 15): where a cleaned test
text matches a train group in which **every** occurrence carries the same label,
that label is used. Groups with conflicting labels are skipped entirely. The
lookup is built from train rows only, so no test label ever influences it.

In [7]:
# --- 8. duplicate-text override (verbatim from Submission 3's notebook) ------
URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
WS_RE = re.compile(r"\s+", flags=re.UNICODE)


def clean_text(t):
    """URL -> ' URL ', @mention -> ' USER ', collapse whitespace, strip."""
    s = "" if t is None else str(t)
    s = URL_RE.sub(" URL ", s)
    s = MENTION_RE.sub(" USER ", s)
    return WS_RE.sub(" ", s).strip()


def unanimous_lookup(texts, labels):
    """cleaned text -> label for groups where EVERY occurrence agrees."""
    df = pd.DataFrame({"t": pd.Series(list(texts), dtype="object"),
                       "y": np.asarray(labels)})
    g = df.groupby("t")["y"].agg(["nunique", "first"])
    uni = g[g["nunique"] == 1]
    lookup = {str(t): int(v) for t, v in uni["first"].items()}
    n_conflicting = int((g["nunique"] > 1).sum())
    return lookup, n_conflicting


def apply_override(texts, pred, lookup):
    out = np.asarray(pred).copy()
    hits = 0
    for i, t in enumerate(texts):
        v = lookup.get(str(t))
        if v is not None:
            out[i] = v
            hits += 1
    return out, hits


clean_tr = train["text"].map(clean_text).to_numpy()
clean_te = test["text"].map(clean_text).to_numpy()
lookup, n_conflict = unanimous_lookup(clean_tr, y)
print("unanimous train groups: %d   conflicting groups skipped: %d"
      % (len(lookup), n_conflict))

unanimous train groups: 47142   conflicting groups skipped: 53


## 6. ID ablation - the features that carry the score

Retraining the *same* frozen policy with the raw `id` dropped, and with the whole
ID-neighbour block dropped, shows how much of the score is ID structure. These
are single-seed diagnostics on the frozen configuration; they are not tuned.

In [8]:
# --- 9. ID ablation on the frozen policy (single seed, diagnostic only) ------
def ablate(drop, seed=42):
    cols = [c for c in Ftr.columns if c not in drop]
    params = dict(PARAMS, seed=seed, bagging_seed=seed, feature_fraction_seed=seed)
    oof = np.zeros((n_tr, 3))
    for k in range(N_SPLITS):
        trm = folds != k
        va = folds == k
        m = lgb.train(params, lgb.Dataset(Ftr.loc[trm, cols], y[trm], weight=WTS[trm]), ROUNDS)
        oof[va] = m.predict(Ftr.loc[va, cols])
    return macro_f1(y, oof.argmax(1))


print("ablation (seed 42, no class correction, frozen params)")
print("  full 60 features (seed 42) : %.6f" % SEED_SCORES["42"])
print("  drop raw id (rid)          : %.6f" % ablate(["rid"]))
print("  drop all 32 ID-neighbour   : %.6f" % ablate(NB_COLS))

ablation (seed 42, no class correction, frozen params)
  full 60 features (seed 42) : 0.693895


  drop raw id (rid)          : 0.689608


  drop all 32 ID-neighbour   : 0.685701


In [9]:
# --- 10. text-only ensemble and the per-source table (disclosure) -----------
text_only_cols = ["%s_p%d" % (n, k) for n in ("tfidf", "knn", "e5") for k in range(3)]
text_only = Ftr[text_only_cols].to_numpy().reshape(-1, 3, 3).mean(axis=1)
print("text-only ensemble (tfidf+knn+e5, equal weight) OOF: %.6f"
      % macro_f1(y, text_only.argmax(1)))
for nm, key in [("lexical tfidf", "tfidf"), ("text knn", "knn"),
                ("semantic e5", "e5"), ("plain id prior", "pid"),
                ("SID id prior", "sid"), ("xlmr id-head", "idh")]:
    z = np.load(SRC[key], allow_pickle=True)
    print("  %-16s %.6f" % (nm, macro_f1(y, z["oof"].argmax(1))))
print("  Submission 3 (text blend)  0.682311")
print("  Submission 4 (this)        %.6f" % POOLED)

text-only ensemble (tfidf+knn+e5, equal weight) OOF: 0.560537
  lexical tfidf    0.516093
  text knn         0.487078
  semantic e5      0.521535
  plain id prior   0.613631
  SID id prior     0.627895
  xlmr id-head     0.648743
  Submission 3 (text blend)  0.682311
  Submission 4 (this)        0.703904


## 7. Test prediction, validation and CSV

The test prediction is `argmax(log(mean test probs over the 3 seeds) + final
offset)`, then the duplicate override. The CSV is written by this notebook and
validated both before writing and after reading it back from disk.

In [10]:
# --- 11. test prediction, pre-write validation, write, post-read validation --
def resolve_out_dir():
    """If this notebook lives inside a uap_submission/reproducible/ tree, write
    to that project's uap_submission/submission/. Otherwise (e.g. a clean-room
    copy) write to the notebook's own directory - never walk up into some other
    project's tree."""
    here = HERE.resolve()
    for base in [here] + list(here.parents):
        cand = base / "uap_submission" / "reproducible"
        if cand.is_dir() and (cand == here or cand in here.parents):
            return base / "uap_submission" / "submission"
    return here


OUT_DIR = resolve_out_dir()
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "submission_4.csv"

test_pred = (log_tst + np.asarray(FINAL_OFFSET, float)).argmax(1)
share_before = [float((test_pred == c).mean()) for c in CLASSES]
train_share = [float((y == c).mean()) for c in CLASSES]
print("test share before override:", [round(v, 4) for v in share_before])
print("train class share          :", [round(v, 4) for v in train_share])
drift_pp = [100.0 * abs(share_before[c] - train_share[c]) for c in CLASSES]
print("drift (pp)                 :", [round(v, 3) for v in drift_pp])
assert all(v <= 6.0 for v in drift_pp), "test class share drifted too far"

test_final, n_hits = apply_override(clean_te, test_pred, lookup)
changed = int((test_final != test_pred).sum())
share_after = [float((test_final == c).mean()) for c in CLASSES]
print("override: %d rows matched, %d labels changed" % (n_hits, changed))
print("test share after override :", [round(v, 4) for v in share_after])

sub = pd.DataFrame({"id": tids, "y_pred": test_final.astype(int)})
# ---- pre-write validation ----
assert list(sub.columns) == ["id", "y_pred"], list(sub.columns)
assert len(sub) == 11955, len(sub)
assert sub["id"].is_unique, "duplicate ids"
assert np.array_equal(sub["id"].to_numpy(), sample["id"].to_numpy()), (
    "id order differs from sample_submission")
assert not sub.isna().any().any(), "null values"
assert sub["y_pred"].dtype.kind in "iu", "y_pred must be integer"
assert set(sub["y_pred"].unique()) <= {0, 1, 2}, "unexpected label"
print("PRE-WRITE validation OK: columns, 11955 rows, unique ids, order matches")
print("sample_submission, no nulls, int dtype, labels in {0,1,2}")

sub.to_csv(OUT_CSV, index=False)

# ---- post-read validation ----
back = pd.read_csv(OUT_CSV)
assert list(back.columns) == ["id", "y_pred"], list(back.columns)
assert len(back) == 11955, len(back)
assert back["id"].is_unique
assert np.array_equal(back["id"].to_numpy(), sample["id"].to_numpy())
assert not back.isna().any().any()
assert back["y_pred"].dtype.kind in "iu"
assert set(back["y_pred"].unique()) <= {0, 1, 2}
assert np.array_equal(back["y_pred"].to_numpy(), sub["y_pred"].to_numpy())
counts = np.bincount(back["y_pred"], minlength=3)
print("POST-READ validation OK: file readable, not truncated, content identical")
print("class counts:", counts.tolist())
print("class shares:", [round(float(v), 4) for v in counts / len(back)])
print("csv bytes   :", OUT_CSV.stat().st_size)
print("csv sha256  :", sha256_of(OUT_CSV))
print("wrote", OUT_CSV)

test share before override: [0.4811, 0.4046, 0.1143]
train class share          : [0.4968, 0.3798, 0.1233]
drift (pp)                 : [1.576, 2.476, 0.9]
override: 272 rows matched, 37 labels changed
test share after override : [0.4796, 0.4059, 0.1144]
PRE-WRITE validation OK: columns, 11955 rows, unique ids, order matches
sample_submission, no nulls, int dtype, labels in {0,1,2}
POST-READ validation OK: file readable, not truncated, content identical
class counts: [5734, 4853, 1368]
class shares: [0.4796, 0.4059, 0.1144]
csv bytes   : 105398
csv sha256  : bfb068893f3245811760d42134253e51a0e5e77004e41a4c27630f2eaaca463f
wrote D:\Desktop\UAP\repro_s4\fresh\submission_4.csv


In [11]:
# --- 12. final summary ------------------------------------------------------
TARGET = CFG["reproduced"]["recorded_target"]
print("=" * 66)
print("Submission 4 - final summary (ID-DERIVED performance stacker)")
print("=" * 66)
print("validation classification :", CFG["validation_classification"])
print("reproduced pooled OOF     : %.6f" % POOLED)
print("recorded target           : %.6f" % TARGET)
print("difference                : %+.6f" % (POOLED - TARGET))
print("within 0.001 of target    :", abs(POOLED - TARGET) <= 0.001)
print("per-class F1              :", [round(float(v), 6) for v in per_class])
print("accuracy                  : %.6f" % ACC)
print("fold scores               :", [round(v, 6) for v in folds_scores])
print("fold std                  : %.6f" % float(np.std(folds_scores)))
print("seed scores (no corr.)    :", {k: round(v, 6) for k, v in SEED_SCORES.items()})
print("frozen class correction   : fold offsets", FOLD_OFFSETS)
print("                            test offset ", FINAL_OFFSET)
print("test class distribution   :", [round(v, 4) for v in share_after])
print("=" * 66)

Submission 4 - final summary (ID-DERIVED performance stacker)
validation classification : STANDARD_OOF_STACKING_WITH_DEPENDENCY
reproduced pooled OOF     : 0.703904
recorded target           : 0.704001
difference                : -0.000097
within 0.001 of target    : True
per-class F1              : [0.782266, 0.67259, 0.656856]
accuracy                  : 0.724261
fold scores               : [0.709531, 0.705706, 0.699544, 0.700173, 0.704385]
fold std                  : 0.003690
seed scores (no corr.)    : {'42': 0.693895, '7': 0.694031, '2026': 0.692881}
frozen class correction   : fold offsets [[0.0, -0.05, -0.6], [0.0, -0.0, -0.55], [0.0, -0.0, -0.55], [0.0, -0.0, -0.5], [0.0, -0.1, -0.65]]
                            test offset  [0.0, -0.0, -0.55]
test class distribution   : [0.4796, 0.4059, 0.1144]
